# L_0.2.2 — 引擎遷移:notebook-primary 重構

> 類別:exp-notebook | 狀態:frozen(執行通過後) | 版本:L_0.2.2(patch:重構,無機制變更) | 日期:2026-07-04
> 依 `docs/01_sop.md` §9:自 L_0.2.2 起一版一 notebook;`src/labor_sim/` 與 `scripts/` 凍結為 legacy。
> **與上一版差異**:零——本版唯一目的是把 v4.1 引擎(至 L_0.2.1 狀態)內嵌為自足 notebook。
> **驗收**:三情境的 regime 與五項指標,與 `results/L_0.2.1/summary.json` 一致(容差 1e-9)。

### 1. 設定
目的:版本字串、輸出路徑、CJK 字型、最小輸出 helper(CSV / summary.json / manifest.json,沿 01_sop §5)。
驗證:cell 末印出 VERSION 與 ROOT;`results/L_0.2.2/` 目錄建立成功。

In [1]:
from __future__ import annotations
import csv, json, os, subprocess, time
from dataclasses import dataclass, field
import numpy as np
import matplotlib
matplotlib.use("Agg")                     # headless,凍結三閘門 9.5
import matplotlib.pyplot as plt
from matplotlib import font_manager

VERSION = "L_0.2.2"
ROOT = (os.path.abspath(os.path.join(os.getcwd(), ".."))
        if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
RESULTS = os.path.join(ROOT, "results", VERSION)
os.makedirs(os.path.join(RESULTS, "figures"), exist_ok=True)
os.makedirs(os.path.join(RESULTS, "data"), exist_ok=True)

_avail = {f.name for f in font_manager.fontManager.ttflist}
for _name in ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC", "SimHei"]:
    if _name in _avail:
        matplotlib.rcParams["font.sans-serif"] = [_name]
        break
matplotlib.rcParams["axes.unicode_minus"] = False


def save_csv(name, columns):
    """圖必配 CSV:等長欄位,NaN 留空。"""
    keys = list(columns)
    cols = [np.asarray(columns[k], float).ravel() for k in keys]
    assert len({c.size for c in cols}) == 1, "欄位長度必須一致"
    with open(os.path.join(RESULTS, "data", name), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(keys)
        for i in range(cols[0].size):
            w.writerow(["" if np.isnan(c[i]) else f"{c[i]:.6g}" for c in cols])


def _merge_json(path, section, payload):
    data = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
    data.setdefault("_version", VERSION)
    data[section] = payload
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.write("\n")


def update_summary(section, payload):
    _merge_json(os.path.join(RESULTS, "summary.json"), section, payload)


def record_manifest(name, **entry):
    try:
        g = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                           capture_output=True, text=True, timeout=5).stdout.strip() or "unknown"
    except Exception:
        g = "unknown"
    _merge_json(os.path.join(RESULTS, "manifest.json"), name,
                {"ran_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 "git_hash": g, **entry})


print(f"VERSION={VERSION}  ROOT={ROOT}")

VERSION=L_0.2.2  ROOT=D:\Project\sideProject\labor_market_simulation


### 2.1 引擎——指標函數
目的:Gini(全體,含失業者 0)與頂端 10% 份額。
驗證:§3 以手算小例 assert(全 0 → 0;完全集中 → Gini→1、top10=1)。
Pseudocode:
```
gini    = 排序後 (2·Σ i·x_i)/(n·Σx) − (n+1)/n      # 值域 [0,1)
top10   = 排序後最大 10% 的收入 / 總收入
```

In [2]:
def gini(x: np.ndarray) -> float:
    """Gini 係數,定義在所有人(含失業者,收入 0)上。"""
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    idx = np.arange(1, n + 1)
    return float((2.0 * np.sum(idx * x) / (n * np.sum(x))) - (n + 1.0) / n)


def top_share(x: np.ndarray, q: float = 0.10) -> float:
    """頂端 q 比例的人賺走多少份額(衡量集中度)。"""
    x = np.asarray(x, dtype=float)
    total = x.sum()
    if total <= 0:
        return 0.0
    k = max(1, int(round(x.size * q)))
    return float(np.sort(x)[-k:].sum() / total)

### 2.2 引擎——參數與職業層級
目的:`Params`(全部旋鈕,基線同 `20_spec_v4` §2)與職業層級切線(σ 的投影標籤)。
驗證:預設值即 spec 基線;L_0.2.1 地基參數(`tasks_per_worker`/`ability_ceiling`)預設 None = 舊行為,向後相容。

In [3]:
@dataclass
class Params:
    n_workers: int = 2000
    task_density: float = 2000.0  # 每單位 σ 的任務數(固定密度)→ 任務數隨 σ_max 等比成長
    steps: int = 240              # 月 × 20 年

    # --- 宏觀:前沿速度 / 任務生成速度 ---
    F0: float = 0.20
    r: float = 0.003
    sigma_max0: float = 1.0
    c: float = 0.003

    # --- 分配:能力尾巴 × 價值凸度 ---
    ability_sigma: float = 0.5
    ability_median: float = 0.5
    V_curvature: float = 2.0

    # --- 其他 ---
    frontier_noise: float = 0.03
    human_learning: float = 0.0   # g:宏觀體制決定者(spec §3.1)
    retrain_rate: float = 0.0

    # --- L_0.2.1 地基(預設 None = 舊行為,向後相容) ---
    tasks_per_worker: float | None = None
    ability_ceiling: float | None = None

    seed: int = 0


OCC_EDGES = (0.4, 0.8, 1.2)
OCC_LABELS = ("失業", "低階 σ<0.4", "中階 0.4–0.8", "高階 0.8–1.2", "精英 σ>1.2")

### 2.3 引擎——能力分佈與排序匹配(機制 cell)
目的:LogNormal 能力生成;貪婪排序匹配(集中若出現,是「排序+重尾+凸V」的後果,不是設定——spec §3.3)。
驗證:§3 assert 每人至多一任務、收入非負;§4 回歸驗證整體行為。
Pseudocode:
```
a ~ ability_median · LogNormal(0, ability_sigma)     # 中位數錨定,尾巴可掃描
任務 σ 由大到小;worker a 由大到小;指標 wi 掃 worker
for 每個任務(大→小):
    if 最強未配者能做(a[wi] ≥ σ):配對,earnings = σ^k,wi += 1
    else:全場無人能做此任務 → 留空,換下一個較小任務
不變量:最強者拿最大可做任務;a < 全部任務者失業(earnings=0)
```

In [4]:
def _make_ability(p: Params, rng) -> np.ndarray:
    """能力分佈:lognormal,中位數錨定 ability_median,使 F(約 0.2→0.9)掃過分佈主體。"""
    return p.ability_median * rng.lognormal(mean=0.0, sigma=p.ability_sigma, size=p.n_workers)


def _match(a: np.ndarray, sigma_tasks: np.ndarray, k: float):
    """排序匹配(assortative,貪婪,O(n log n))。
    回傳 (earnings, assigned_sigma, n_filled);未配到者 earnings=0、assigned=-1。"""
    n = a.size
    earnings = np.zeros(n)
    assigned = np.full(n, -1.0)
    if sigma_tasks.size == 0:
        return earnings, assigned, 0

    tasks_desc = np.sort(sigma_tasks)[::-1]
    order = np.argsort(a)[::-1]
    a_desc = a[order]

    wi = 0
    for tsig in tasks_desc:
        if wi >= n:
            break
        if a_desc[wi] >= tsig:
            earnings[order[wi]] = tsig ** k
            assigned[order[wi]] = tsig
            wi += 1
    return earnings, assigned, wi

### 2.4 引擎——主迴圈與體制分類(機制 cell)
目的:`run_sim`(一條規則 + 回饋 + 全部指標)與 `classify_regime`(order parameter 客觀分類)。
驗證:§3 不變量(值域/單調/同 seed 重現/飽和天花板);§4 重現 L_0.2.1。
Pseudocode:
```
for t in 0..steps:
    F = F0 + r·t;  σ_max = σ_max0 + c·t
    生成任務 σ ~ U[0, σ_max](固定密度 → 數量隨 σ_max 成長)
    門檻 = F + N(0, ε)                        # 毛邊前沿
    σ < 門檻 → AI 拿走;其餘 → 排序匹配給人
    記錄:就業率/Gini(全體+在職者)/top10/人類區/職業結構/技能演化
    在崗者以 g、失業者以 retrain_rate 成長;有 ceiling → 飽和式 a += g·(ceiling−a)
末段 order parameter:人類區關閉或就業腰斬 → collapse;top10>0.5 → concentration;否則 reallocation
```

In [5]:
@dataclass
class Result:
    params: Params
    F: np.ndarray
    sigma_max: np.ndarray
    employment_rate: np.ndarray
    gini: np.ndarray
    top10_share: np.ndarray
    human_zone: np.ndarray
    final_ability: np.ndarray
    gini_employed: np.ndarray = field(default=None, repr=False)
    regime: str = ""
    history_earnings_last: np.ndarray = field(default=None, repr=False)
    occ_shares: np.ndarray = field(default=None, repr=False)
    ai_substitution: np.ndarray = field(default=None, repr=False)
    new_job_rate: np.ndarray = field(default=None, repr=False)
    mean_ability: np.ndarray = field(default=None, repr=False)
    mean_ability_emp: np.ndarray = field(default=None, repr=False)
    retrain_success: np.ndarray = field(default=None, repr=False)
    demand_median: np.ndarray = field(default=None, repr=False)
    supply_median: np.ndarray = field(default=None, repr=False)
    unmet_demand: np.ndarray = field(default=None, repr=False)
    unused_supply: np.ndarray = field(default=None, repr=False)


def run_sim(p: Params | None = None, **kw) -> Result:
    if p is None:
        p = Params(**kw)
    rng = np.random.default_rng(p.seed)

    a = _make_ability(p, rng)

    # L_0.2.1 槽位校準:tasks_per_worker → 反推 task_density 使 t=0 人類任務數 = ratio × n_workers
    task_density = p.task_density
    if p.tasks_per_worker is not None:
        span0 = max(1e-9, p.sigma_max0 - p.F0)
        task_density = p.tasks_per_worker * p.n_workers / span0

    S = p.steps
    F_hist = np.empty(S); smax_hist = np.empty(S)
    emp_hist = np.empty(S); gini_hist = np.empty(S)
    gini_emp_hist = np.empty(S)
    top_hist = np.empty(S); zone_hist = np.empty(S)
    occ_hist = np.zeros((S, 5))
    ai_sub_hist = np.empty(S); newjob_hist = np.empty(S)
    mean_a_hist = np.empty(S); mean_a_emp_hist = np.empty(S)
    retrain_hist = np.zeros(S)
    demand_hist = np.empty(S); supply_hist = np.empty(S)
    unmet_hist = np.empty(S); unused_hist = np.empty(S)

    last_earn = np.zeros(p.n_workers)
    prev_emp_mask = np.zeros(p.n_workers, dtype=bool)
    k = p.V_curvature

    for t in range(S):
        F = p.F0 + p.r * t
        sigma_max = p.sigma_max0 + p.c * t

        n_tasks_t = max(1, int(round(task_density * sigma_max)))
        sigma = rng.uniform(0.0, sigma_max, size=n_tasks_t)

        thresh = F + rng.normal(0.0, p.frontier_noise, size=n_tasks_t)
        ai_mask = sigma < thresh
        human_tasks = sigma[~ai_mask]

        earn, assigned, n_filled = _match(a, human_tasks, k)
        last_earn = earn
        emp_mask = earn > 0

        F_hist[t] = F
        smax_hist[t] = sigma_max
        emp_hist[t] = float(emp_mask.mean())
        gini_hist[t] = gini(earn)
        gini_emp_hist[t] = gini(earn[emp_mask]) if emp_mask.any() else np.nan
        top_hist[t] = top_share(earn, 0.10)
        zone_hist[t] = max(sigma_max - F, 0.0)

        occ_hist[t, 0] = 1.0 - emp_mask.mean()
        if emp_mask.any():
            tiers = np.digitize(assigned[emp_mask], OCC_EDGES)
            for i in range(4):
                occ_hist[t, i + 1] = np.count_nonzero(tiers == i) / p.n_workers
        all_val = np.sum(sigma ** k)
        ai_sub_hist[t] = float(np.sum(sigma[ai_mask] ** k) / all_val) if all_val > 0 else 0.0
        newjob_hist[t] = float(np.sum(sigma[sigma > p.sigma_max0] ** k) / all_val) if all_val > 0 else 0.0

        mean_a_hist[t] = float(a.mean())
        mean_a_emp_hist[t] = float(a[emp_mask].mean()) if emp_mask.any() else np.nan
        if t > 0 and (~prev_emp_mask).any():
            recovered = np.count_nonzero(emp_mask & ~prev_emp_mask)
            retrain_hist[t] = recovered / np.count_nonzero(~prev_emp_mask)
        demand_hist[t] = float(np.median(human_tasks)) if human_tasks.size else F
        supply_hist[t] = float(np.median(a))
        unmet_hist[t] = float(1.0 - n_filled / human_tasks.size) if human_tasks.size else 0.0
        unused_hist[t] = occ_hist[t, 0]

        prev_emp_mask = emp_mask

        if p.human_learning > 0 or p.retrain_rate > 0:
            growth = np.where(emp_mask, p.human_learning, p.retrain_rate)
            if p.ability_ceiling is not None:
                a = a + growth * (p.ability_ceiling - a)   # 飽和:趨近 ceiling、永不超過
            else:
                a = a * (1.0 + growth)                     # 舊:無上限複利

    res = Result(
        params=p,
        F=F_hist, sigma_max=smax_hist,
        employment_rate=emp_hist, gini=gini_hist,
        top10_share=top_hist, human_zone=zone_hist,
        final_ability=a, history_earnings_last=last_earn,
        gini_employed=gini_emp_hist,
        occ_shares=occ_hist, ai_substitution=ai_sub_hist, new_job_rate=newjob_hist,
        mean_ability=mean_a_hist, mean_ability_emp=mean_a_emp_hist,
        retrain_success=retrain_hist,
        demand_median=demand_hist, supply_median=supply_hist,
        unmet_demand=unmet_hist, unused_supply=unused_hist,
    )
    res.regime = classify_regime(res)
    return res


def classify_regime(res: Result) -> str:
    """order parameter 客觀分類,不靠眼睛看。"""
    emp = res.employment_rate
    tail = emp[-12:].mean()
    head = emp[:12].mean()
    zone_closed = res.human_zone[-1] <= 1e-9
    top = res.top10_share[-12:].mean()

    if zone_closed or (head > 0 and tail < 0.5 * head) or tail < 0.25:
        return "collapse"
    if top > 0.50:
        return "concentration"
    return "reallocation"

### 3. 引擎驗證(不變量)
目的:用 assert 固定引擎的基本性質——錯一條就不准往下跑。
驗證項:指標手算小例;就業率值域 [0,1];F 嚴格遞增;收入非負;飽和學習不破天花板(seed=0 實測);同 seed 逐位重現。

In [6]:
# 指標:手算小例
assert gini(np.zeros(5)) == 0.0
assert abs(gini(np.array([0, 0, 0, 0, 10.0])) - 0.8) < 1e-12   # 完全集中(n=5): (n-1)/n
assert top_share(np.array([1.0] * 9 + [91.0])) == 0.91

_p = Params(r=0.004, human_learning=0.004, tasks_per_worker=1.0, ability_ceiling=3.0, seed=0)
_res = run_sim(_p)
assert np.all((_res.employment_rate >= 0) & (_res.employment_rate <= 1))
assert np.all(np.diff(_res.F) > 0)                      # 前沿嚴格遞增
assert _res.history_earnings_last.min() >= 0
assert _res.final_ability.max() <= 3.0 + 1e-9           # 飽和學習不破天花板(seed=0)

_res2 = run_sim(Params(r=0.004, human_learning=0.004, tasks_per_worker=1.0, ability_ceiling=3.0, seed=0))
assert np.array_equal(_res.employment_rate, _res2.employment_rate)   # 同 seed 逐位重現

print("引擎不變量:全部通過")

引擎不變量:全部通過


### 4. 回歸驗證(凍結閘門 9.4-2):重現 L_0.2.1
目的:證明遷移**零漂移**——同參數同 seed 下,三情境的 regime 與五項指標必須與凍結的 `results/L_0.2.1/summary.json` 一致。
驗證:相對容差 1e-9(同程式路徑 + 鎖定的 numpy 版本,理應逐位一致);任何一項不合立即 AssertionError。

In [7]:
CALIB = dict(tasks_per_worker=1.0, ability_ceiling=3.0)
SCENARIOS = {
    "崩潰 r≫g (無學習)": dict(r=0.005, human_learning=0.000),
    "拉鋸 r≈g":          dict(r=0.004, human_learning=0.004),
    "重配置 g≥r":        dict(r=0.003, human_learning=0.006),
}
METRICS = ["employment_t0", "employment_final", "gini_all_final",
           "gini_employed_final", "top10_final"]

with open(os.path.join(ROOT, "results", "L_0.2.1", "summary.json"), encoding="utf-8") as f:
    ref = json.load(f)["baseline_v021"]

results = {}
for name, kw in SCENARIOS.items():
    res = run_sim(Params(**kw, **CALIB))
    results[name] = res
    got = {
        "employment_t0": float(res.employment_rate[0]),
        "employment_final": float(res.employment_rate[-1]),
        "gini_all_final": float(res.gini[-1]),
        "gini_employed_final": float(res.gini_employed[-1]),
        "top10_final": float(res.top10_share[-1]),
    }
    exp = ref[name]
    assert res.regime == exp["regime"], (name, res.regime, exp["regime"])
    for m in METRICS:
        tol = 1e-9 * max(1.0, abs(exp[m]))
        assert abs(got[m] - exp[m]) <= tol, (name, m, got[m], exp[m])
    print(f"{name:14s} regime={res.regime:13s} emp_end={got['employment_final']:.3f}  ✓ 與 L_0.2.1 一致")

print("回歸閘門:通過(遷移零漂移)")

崩潰 r≫g (無學習)   regime=collapse      emp_end=0.021  ✓ 與 L_0.2.1 一致


拉鋸 r≈g         regime=reallocation  emp_end=0.699  ✓ 與 L_0.2.1 一致


重配置 g≥r        regime=reallocation  emp_end=0.973  ✓ 與 L_0.2.1 一致
回歸閘門:通過(遷移零漂移)


### 5. 圖與輸出:三情境時間序列
目的:重繪 L_0.2.1 基線五面板(本版產物,存 `results/L_0.2.2/`);圖必配 CSV + summary + manifest(閘門 9.4-3)。
驗證:圖檔與 `data/baseline_v022.csv` 生成;summary 寫入 `baseline_v022` 一節。

In [8]:
PANELS = [
    ("employment_rate", "就業率 employment"),
    ("human_zone",      "人類保護區寬度 σ_max−F"),
    ("gini",            "全體 Gini(含失業者)"),
    ("gini_employed",   "在職者 Gini(只算 earnings>0)"),
    ("top10_share",     "頂端 10% 收入份額"),
]

months = np.arange(next(iter(results.values())).employment_rate.size)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
flat = axes.ravel()
for ax, (attr, title) in zip(flat, PANELS):
    for name, res in results.items():
        ax.plot(months, getattr(res, attr), label=name, lw=2)
    ax.set_title(title); ax.set_xlabel("月"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
flat[len(PANELS)].axis("off")
fig.suptitle("L_0.2.2 引擎遷移基線(= L_0.2.1 設定,notebook 引擎)", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "figures", "baseline_v022.png"), dpi=120)
plt.show()

cols = {"month": months}
summ = {}
TAG = {"崩潰 r≫g (無學習)": "collapse", "拉鋸 r≈g": "tug", "重配置 g≥r": "realloc"}
for name, res in results.items():
    for attr, _ in PANELS:
        cols[f"{TAG[name]}__{attr}"] = getattr(res, attr)
    summ[name] = {
        "regime": res.regime,
        "employment_t0": float(res.employment_rate[0]),
        "employment_final": float(res.employment_rate[-1]),
        "gini_all_final": float(res.gini[-1]),
        "gini_employed_final": float(res.gini_employed[-1]),
        "top10_final": float(res.top10_share[-1]),
    }
save_csv("baseline_v022.csv", cols)
update_summary("baseline_v022", summ)
update_summary("regression_vs_L_0.2.1", {"passed": True, "tolerance": "rel 1e-9", "metrics": METRICS})
record_manifest("L_0.2.2.ipynb",
                params={"CALIB": CALIB, "SCENARIOS": SCENARIOS},
                seeds=[0],
                notes="引擎遷移(notebook-primary,無機制變更);回歸驗證對象=results/L_0.2.1/summary.json")
print("產物已寫入", RESULTS)

C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:18: UserWarning: Glyph 8811 (\N{MUCH GREATER-THAN}) missing from font(s) Microsoft JhengHei.
  fig.tight_layout()
C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:18: UserWarning: Glyph 8776 (\N{ALMOST EQUAL TO}) missing from font(s) Microsoft JhengHei.
  fig.tight_layout()
C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:18: UserWarning: Glyph 8805 (\N{GREATER-THAN OR EQUAL TO}) missing from font(s) Microsoft JhengHei.
  fig.tight_layout()
C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:18: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Microsoft JhengHei.
  fig.tight_layout()


C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:19: UserWarning: Glyph 8811 (\N{MUCH GREATER-THAN}) missing from font(s) Microsoft JhengHei.
  fig.savefig(os.path.join(RESULTS, "figures", "baseline_v022.png"), dpi=120)
C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:19: UserWarning: Glyph 8776 (\N{ALMOST EQUAL TO}) missing from font(s) Microsoft JhengHei.
  fig.savefig(os.path.join(RESULTS, "figures", "baseline_v022.png"), dpi=120)
C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:19: UserWarning: Glyph 8805 (\N{GREATER-THAN OR EQUAL TO}) missing from font(s) Microsoft JhengHei.
  fig.savefig(os.path.join(RESULTS, "figures", "baseline_v022.png"), dpi=120)
C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:19: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Microsoft JhengHei.
  fig.savefig(os.path.join(RESULTS, "figures", "baseline_v022.png"), dpi=120)


產物已寫入 D:\Project\sideProject\labor_market_simulation\results\L_0.2.2


C:\Users\user\AppData\Local\Temp\ipykernel_43824\3266479586.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6. Headline
目的:本版 ≤3 條含數字的結論,餵 `30_exp_ledger`。

In [9]:
print("1. 遷移零漂移:三情境 regime(collapse/reallocation/reallocation)與五項指標")
print("   全部與 L_0.2.1 一致(相對容差 1e-9)。")
print("2. 自 L_0.3.0 起 notebook-primary:src/labor_sim 與 scripts/ 凍結為 legacy。")
print("3. 本版無新機制、無新結論(patch);不另開 30_exp 檢討。")

1. 遷移零漂移:三情境 regime(collapse/reallocation/reallocation)與五項指標
   全部與 L_0.2.1 一致(相對容差 1e-9)。
2. 自 L_0.3.0 起 notebook-primary:src/labor_sim 與 scripts/ 凍結為 legacy。
3. 本版無新機制、無新結論(patch);不另開 30_exp 檢討。
